In [ ]:
# ============================================================
# 2023.01 ~ 2025.12 동안 공백이 한 번도 없는 법인 추출
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. 엑셀 불러오기
     ***엑셀 파일 이름을 im_bank_data.xlsx로 가정, 실행 코드 파일과 엑셀 파일이 같은 폴더 위치에 있다고 가정
# ------------------------------------------------------------

FILE_PATH = "im_bank_data.xlsx"

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Sheet1",
    engine="openpyxl"
)

print("===== 원본 데이터 =====")
print(f"행 수: {len(df):,}")
print(f"열 수: {len(df.columns):,}")
print(f"고유 법인 수: {df['법인ID'].nunique():,}")


# ------------------------------------------------------------
# 2. 기준년월 정리
# ------------------------------------------------------------

df["기준년월"] = pd.to_numeric(
    df["기준년월"],
    errors="raise"
).astype(int)

print("\n===== 기간 확인 =====")
print("최소 기준년월:", df["기준년월"].min())
print("최대 기준년월:", df["기준년월"].max())
print("전체 고유 월 수:", df["기준년월"].nunique())


# ------------------------------------------------------------
# 3. 분석기간 36개월 생성
#    2023.01 ~ 2025.12
# ------------------------------------------------------------

expected_months = (
    pd.period_range(
        start="2023-01",
        end="2025-12",
        freq="M"
    )
    .strftime("%Y%m")
    .astype(int)
)

print("\n===== 기대 기간 =====")
print("시작:", expected_months[0])
print("종료:", expected_months[-1])
print("개월 수:", len(expected_months))


# ------------------------------------------------------------
# 4. 법인별 관측 개월 수 계산
# ------------------------------------------------------------

corp_month_count = (
    df.groupby("법인ID")["기준년월"]
      .nunique()
)

print("\n===== 법인별 관측개월 수 =====")
print(f"최소: {corp_month_count.min()}")
print(f"중앙값: {corp_month_count.median():.1f}")
print(f"최대: {corp_month_count.max()}")


# ------------------------------------------------------------
# 5. 36개월 모두 관측된 법인 추출
# ------------------------------------------------------------

complete_corp_ids = corp_month_count[
    corp_month_count == 36
].index

# 해당 법인들의 행만 추출
complete_df = (
    df[df["법인ID"].isin(complete_corp_ids)]
    .copy()
    .sort_values(["법인ID", "기준년월"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. 최종 검증
# ------------------------------------------------------------

check_month_count = (
    complete_df.groupby("법인ID")["기준년월"]
               .nunique()
)

print("\n===== 공백이 한 번도 없는 법인 =====")
print(f"법인 수: {complete_df['법인ID'].nunique():,}")
print(f"행 수: {len(complete_df):,}")

print("\n===== 검증 =====")
print(
    "모든 법인이 정확히 36개월:",
    "PASS" if check_month_count.eq(36).all() else "CHECK"
)

print(
    "기대 법인 수 3,372:",
    "PASS" if complete_df["법인ID"].nunique() == 3372 else "CHECK"
)

print(
    "기대 행 수 121,392:",
    "PASS" if len(complete_df) == 3372 * 36 else "CHECK"
)


# ------------------------------------------------------------
# 7. 실제로 월 공백이 없는지도 최종 확인
# ------------------------------------------------------------

expected_month_set = set(expected_months)

all_months_complete = (
    complete_df.groupby("법인ID")["기준년월"]
    .apply(lambda x: set(x) == expected_month_set)
    .all()
)

print(
    "2023.01~2025.12 월이 하나도 빠지지 않음:",
    "PASS" if all_months_complete else "CHECK"
)

===== 원본 데이터 =====
행 수: 365,988
열 수: 70
고유 법인 수: 15,473

===== 기간 확인 =====
최소 기준년월: 202301
최대 기준년월: 202512
전체 고유 월 수: 36

===== 기대 기간 =====
시작: 202301
종료: 202512
개월 수: 36

===== 법인별 관측개월 수 =====
최소: 1
중앙값: 27.0
최대: 36

===== 공백이 한 번도 없는 법인 =====
법인 수: 3,372
행 수: 121,392

===== 검증 =====
모든 법인이 정확히 36개월: PASS
기대 법인 수 3,372: PASS
기대 행 수 121,392: PASS
2023.01~2025.12 월이 하나도 빠지지 않음: PASS


In [3]:
# ============================================================
# 36개월 완전관측은 아니지만,
# 가지고 있는 모든 년월이 연속된 법인 찾기
# ============================================================

# 기준년월을 월 단위 Period로 변환
temp = df[["법인ID", "기준년월"]].copy()

temp["month_period"] = pd.to_datetime(
    temp["기준년월"].astype(str),
    format="%Y%m"
).dt.to_period("M")


# ------------------------------------------------------------
# 법인별로 관측된 월이 연속인지 판정
# ------------------------------------------------------------

def check_consecutive_months(x):
    months = x.drop_duplicates().sort_values()

    # 월을 연속적인 정수값으로 변환
    month_numbers = months.astype("int64").to_numpy()

    # 1개월만 있어도 '가지고 있는 기간 내에서는 연속'으로 간주
    if len(month_numbers) <= 1:
        return True

    # 모든 인접 월의 차이가 정확히 1이면 연속
    return np.all(np.diff(month_numbers) == 1)


corp_consecutive = (
    temp.groupby("법인ID")["month_period"]
        .apply(check_consecutive_months)
)


# ------------------------------------------------------------
# 법인별 관측개월 수
# ------------------------------------------------------------

corp_month_count = (
    temp.groupby("법인ID")["month_period"]
        .nunique()
)


# ------------------------------------------------------------
# 36개월 완전관측은 제외하고,
# 나머지 중 내부 공백이 없는 법인
# ------------------------------------------------------------

partial_consecutive_mask = (
    (corp_month_count < 36)
    & (corp_consecutive)
)

partial_consecutive_ids = (
    partial_consecutive_mask[
        partial_consecutive_mask
    ].index
)


# 해당 법인 데이터
partial_consecutive_df = (
    df[df["법인ID"].isin(partial_consecutive_ids)]
    .copy()
    .sort_values(["법인ID", "기준년월"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 결과 출력
# ------------------------------------------------------------

print("===== 36개월 미만이지만 관측기간 내 공백 없는 법인 =====")
print(f"법인 수: {len(partial_consecutive_ids):,}")

print("\n===== 참고 =====")
print(f"전체 법인 수: {df['법인ID'].nunique():,}")
print(f"36개월 완전관측 법인: {(corp_month_count == 36).sum():,}")
print(f"36개월 미만 법인: {(corp_month_count < 36).sum():,}")

print(
    "36개월 미만 중 연속 관측 법인:",
    f"{partial_consecutive_mask.sum():,}"
)

print(
    "36개월 미만 중 중간 공백이 있는 법인:",
    f"{((corp_month_count < 36) & (~corp_consecutive)).sum():,}"
)


# ------------------------------------------------------------
# 연속 관측 법인들의 관측개월 수 분포
# ------------------------------------------------------------

print("\n===== 연속 관측 법인의 관측개월 수 분포 =====")

print(
    corp_month_count.loc[partial_consecutive_ids]
                    .value_counts()
                    .sort_index()
)

===== 36개월 미만이지만 관측기간 내 공백 없는 법인 =====
법인 수: 5,685

===== 참고 =====
전체 법인 수: 15,473
36개월 완전관측 법인: 3,372
36개월 미만 법인: 12,101
36개월 미만 중 연속 관측 법인: 5,685
36개월 미만 중 중간 공백이 있는 법인: 6,416

===== 연속 관측 법인의 관측개월 수 분포 =====
month_period
1     319
2     281
3     268
4     230
5     257
6     237
7     196
8     255
9     192
10    142
11    182
12    161
13    134
14    185
15    191
16    188
17    177
18    192
19    121
20    167
21    154
22     98
23    101
24     74
25     81
26    111
27    131
28    113
29    113
30    101
31     95
32    129
33     93
34     67
35    149
Name: count, dtype: int64
